# LLMFS × TabPFN 3차 경량 실험 — 1×5 CV + Inner Holdout

1·2차 코드는 수정하지 않습니다. 3차는 계산량을 줄이기 위해 반복 outer CV를 `1×5`로
축소하고, 각 outer-train 내부에서 stratified holdout 한 번으로 설정을 선택합니다.

- 후보 k: `32/64/128`
- Hybrid alpha: `0.25/0.5/0.75`
- 주 비교: Adaptive Hybrid, 기존 Hybrid, Text, Data-only, MI, LASSO,
  Random 3 seeds, CatBoost-RFE, NoFS
- 공식 평가: 5개 outer-test fold를 합친 pooled OOF
- 통계: paired bootstrap 5,000회, 95% CI, Holm 보정

기본 실행 범위는 **Core TabPFN 330건**입니다. 의미 검증·모델 compatibility·embedding은
이번 경량 실행에서 제외했습니다. 모든 실행 셀은 이미 활성화되어 있습니다.


## 0. 환경과 프로젝트 불러오기

Python 3.11, CUDA PyTorch, TabPFN 8.1.0과 로컬 v2.5 checkpoint를 사용합니다.
노트북을 `3차` 폴더에서 열고 `모두 실행`을 누르세요.


In [1]:
import sys
print(sys.executable)


c:\Users\user\AppData\Local\Programs\Python\Python311\python.exe


In [2]:
from pathlib import Path
import importlib
import json
import os
import sys
import pandas as pd

candidates = [
    Path.cwd().resolve(),
    Path.home() / 'Desktop' / '2026 캡스톤_2' / '2차 실험_학회 포스터' / '3차',
]
PROJECT_ROOT = next((p for p in candidates if (p / 'EXPERIMENT_CODE').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('3차/EXPERIMENT_CODE 폴더를 찾지 못했습니다.')

CODE_ROOT = PROJECT_ROOT / 'EXPERIMENT_CODE'
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

import llmfs_pipeline as exp
exp = importlib.reload(exp)
CFG = exp.load_config(CODE_ROOT / 'experiment_config.json')
PATHS = exp.init_result_dirs(CFG)
exp.seed_everything(int(CFG['seeds']['global_seed']))

print('Pipeline:', exp.PIPELINE_VERSION)
print('Project:', PROJECT_ROOT)
print('Data:', CFG['data_root'])
print('Results:', CFG['results_root'])


Pipeline: 4.1.1
Project: C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3차
Data: C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\DATA
Results: C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3차\RESULTS_LIGHT_1X5_HOLDOUT


In [3]:
env = exp.environment_report(CFG, require_cuda=True)
display(env)
assert env['cuda_available'] is True
assert env['tabpfn_checkpoint']['is_file'] is True
print('CUDA / local TabPFN checkpoint: PASS')


{'created_at': '2026-08-14T05:40:46.790554+00:00',
 'pipeline_version': '4.1.1',
 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]',
 'executable': 'c:\\Users\\user\\AppData\\Local\\Programs\\Python\\Python311\\python.exe',
 'platform': 'Windows-10-10.0.19045-SP0',
 'cuda_available': True,
 'cuda_device': 'NVIDIA GeForce RTX 4060 Ti',
 'torch_cuda_runtime': '13.0',
 'torch_error': None,
 'tabpfn_checkpoint': {'path': 'C:\\Users\\user\\Desktop\\2026 캡스톤_2차\\tabpfn-v2.5-classifier-v2.5_default.ckpt\\tabpfn-v2.5-classifier-v2.5_default.ckpt',
  'exists': True,
  'is_file': True,
  'size_bytes': 42935499},
 'packages': {'numpy': '1.26.4',
  'pandas': '3.0.2',
  'scipy': '1.16.3',
  'scikit-learn': '1.8.0',
  'catboost': '1.2.10',
  'tabpfn': '8.1.0',
  'tabpfn-extensions': '0.5.0',
  'google-genai': '2.17.0',
  'torch': '2.11.0+cu130',
  'umap-learn': '0.5.12'},
 'config_hash': '677e7d670a961887fcab4c15c24492cd9451188f3007495506edca593c4ddf6e'}

CUDA / local TabPFN checkpoint: PASS


## 1. 데이터·1×5 outer CV·내부 holdout 검사

소표본화는 outer-train에만 적용합니다. Inner holdout은 outer-train의 약 75%로 특징을
선택·학습하고 나머지 약 25%로 k·alpha를 고릅니다. Outer-test는 설정 선택에 사용하지 않습니다.


In [4]:
data_summary = exp.validate_all_data(CFG)
mapping_report = exp.semantic_mapping_report(CFG, raise_on_failure=True)
display(data_summary)
display(mapping_report)

assert set(data_summary['dataset']) == {'colon', 'golub', 'metabric'}
assert (data_summary['outer_repeats'] == 1).all()
assert (data_summary['outer_folds'] == 5).all()
assert mapping_report['passed'].all()
assert CFG['nested_tuning']['strategy'] == 'stratified_holdout'
assert int(CFG['nested_tuning']['inner_folds']) == 1
assert CFG['nested_tuning']['k_grid'] == [32, 64, 128]
assert CFG['nested_tuning']['alpha_grid'] == [0.25, 0.5, 0.75]
assert CFG['execution']['enabled_stages'] == ['core']
print('Data / 1×5 outer CV / one inner holdout: PASS')


,dataset,n,p,p_over_n,class_0,class_1,source_outer_repeats,source_outer_folds,outer_repeats,outer_folds,split_strategy,smallest_test_fold
0,golub,72,3051,42.375000,47,25,10,5,1,5,generated_stratified,14
1,colon,62,1991,32.112903,22,40,10,5,1,5,generated_stratified,12
2,metabric,1904,684,0.359244,801,1103,10,5,1,5,generated_stratified,380


,dataset,n_features,real_name_column,n_nonempty_real_names,n_gene_symbols,gene_symbol_coverage,required_minimum,mapping_required,passed
0,golub,3051,semantic_name,3051,1607,0.526713,0.5,True,True
1,colon,1991,semantic_name,1991,517,0.259669,0.2,True,True
2,metabric,684,semantic_name,684,662,0.967836,0.0,False,True


Data / 1×5 outer CV / one inner holdout: PASS


In [5]:
execution_plan = exp.experiment_execution_plan(CFG)
display(execution_plan)

core_grid = exp.build_experiment_grid(CFG, 'core')
assert len(core_grid) == 330
assert int(execution_plan.loc[execution_plan['stage'] == 'TOTAL', 'model_runs'].iloc[0]) == 330
print('Core outer runs:', len(core_grid))
print('Expected inner Logistic fits: 1,260')
print('Paired fold-seed base:', CFG['seeds']['model_seed'])


,stage,model,model_runs
0,core,tabpfn,330
1,TOTAL,all,330


Core outer runs: 330
Expected inner Logistic fits: 1,260
Paired fold-seed base: 20260818


## 2. Gemini score 준비

특징 이름과 task가 같은 2차 LLM 점수를 feature ID와 설정 기준으로 검증해 재사용합니다.
검증 실패 시에만 API key를 입력하고 기존 Gemini Batch를 실행합니다.


In [6]:
RUN_LLM_PREPARE = True
if RUN_LLM_PREPARE:
    llm_cache = exp.reuse_llm_score_cache(CFG)
    display(llm_cache)
    if not llm_cache['ready']:
        display(exp.llm_execution_plan(CFG))
        _gemini_key = exp.ensure_gemini_api_key()
        display(exp.submit_all_llm_jobs(CFG, confirm='SUBMIT_GEMINI_BATCH_JOBS'))
        for condition, runs in [('real', int(CFG['llm']['real_runs'])), ('anonymous', int(CFG['llm']['anonymous_runs']))]:
            for run_index in range(1, runs + 1):
                status = exp.get_llm_job_status(CFG, condition, run_index)
                if status['state'] != 'JOB_STATE_SUCCEEDED':
                    status = exp.wait_for_llm_job(CFG, condition, run_index, timeout_hours=24.0)
                if status['state'] != 'JOB_STATE_SUCCEEDED':
                    raise RuntimeError(f'Gemini Batch failed: {condition}/run{run_index}: {status}')
        display(exp.retrieve_and_prepare_all_llm_scores(CFG))

llm_status = exp.llm_score_cache_status(CFG)
display(llm_status)
assert llm_status['valid'].all()
print('All LLM scores: READY')


{'source': 'current_results', 'copied_files': 0, 'ready': True}

,dataset,condition,exists,valid,n_features,path
0,golub,real,True,True,3051,C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3...
1,golub,anonymous,True,True,3051,C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3...
2,golub,permuted_seed_0007,True,True,3051,C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3...
3,golub,permuted_seed_0019,True,True,3051,C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3...
4,golub,permuted_seed_0031,True,True,3051,C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3...
5,colon,real,True,True,1991,C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3...
6,colon,anonymous,True,True,1991,C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3...
7,colon,permuted_seed_0007,True,True,1991,C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3...
8,colon,permuted_seed_0019,True,True,1991,C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3...
9,colon,permuted_seed_0031,True,True,1991,C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3...


All LLM scores: READY


## 3. 실행 전 smoke test

In [7]:
RUN_SMOKE = True
if RUN_SMOKE:
    smoke_status = exp.run_grid(CFG, exp.build_experiment_grid(CFG, 'smoke'))
    display(smoke_status)
    assert (smoke_status['status'] == 'completed').all()


[1/2] colon mi logistic r1f1 [saved fold reused]
[2/2] colon random logistic r1f1 [saved fold reused]


,stage,dataset,sample_regime,repeat,fold,method,condition,k,alpha,selector_seed,model,status,prediction_path,resume_hit,wall_seconds,error
0,smoke,colon,full,1,1,mi,name_invariant,8,0.5,7,logistic,completed,C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3...,True,0.000003,None
1,smoke,colon,full,1,1,random,name_invariant,8,0.5,7,logistic,completed,C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3...,True,0.000003,None


## 4. 주 분석 — Core TabPFN 330건

모든 방법은 동일한 inner holdout과 k 후보를 사용합니다. Hybrid 두 방식만 alpha 후보를
추가합니다. Inner holdout으로 선택된 설정을 outer-train 전체에 다시 적용하고 outer-test를
TabPFN으로 예측합니다.


In [8]:
RUN_CORE = True
if RUN_CORE:
    # This cell can be rerun directly after an interruption. Reload the patched
    # pipeline and reuse every exact fold prediction already saved on disk.
    exp = importlib.reload(exp)
    CFG = exp.load_config(CODE_ROOT / 'experiment_config.json')
    core_grid = exp.build_experiment_grid(CFG, 'core')
    saved_before = sum(
        exp.prediction_path(CFG, row.where(pd.notna(row), None).to_dict()).is_file()
        for _, row in core_grid.iterrows()
    )
    print(f'Resume mode: {saved_before}/{len(core_grid)} saved folds will not be retrained.')
    core_status = exp.run_grid(CFG, core_grid)
    display(core_status[['status', 'resume_hit']].value_counts(dropna=False))
    assert (core_status['status'] == 'completed').all()


Resume mode: 262/330 saved folds will not be retrained.
[1/330] golub text tabpfn r1f1 [saved fold reused]
[2/330] golub hybrid tabpfn r1f1 [saved fold reused]
[3/330] golub adaptive_hybrid tabpfn r1f1 [saved fold reused]
[4/330] golub data_only tabpfn r1f1 [saved fold reused]
[5/330] golub mi tabpfn r1f1 [saved fold reused]
[6/330] golub lasso tabpfn r1f1 [saved fold reused]
[7/330] golub random tabpfn r1f1 [saved fold reused]
[8/330] golub random tabpfn r1f1 [saved fold reused]
[9/330] golub random tabpfn r1f1 [saved fold reused]
[10/330] golub catboost_rfe tabpfn r1f1 [saved fold reused]
[11/330] golub no_fs tabpfn r1f1 [saved fold reused]
[12/330] golub text tabpfn r1f2 [saved fold reused]
[13/330] golub hybrid tabpfn r1f2 [saved fold reused]
[14/330] golub adaptive_hybrid tabpfn r1f2 [saved fold reused]
[15/330] golub data_only tabpfn r1f2 [saved fold reused]
[16/330] golub mi tabpfn r1f2 [saved fold reused]
[17/330] golub lasso tabpfn r1f2 [saved fold reused]
[18/330] golub rando

c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[222/330] metabric hybrid tabpfn r1f1
[223/330] metabric adaptive_hybrid tabpfn r1f1
[224/330] metabric data_only tabpfn r1f1 [saved fold reused]
[225/330] metabric mi tabpfn r1f1 [saved fold reused]
[226/330] metabric lasso tabpfn r1f1
[227/330] metabric random tabpfn r1f1
[228/330] metabric random tabpfn r1f1
[229/330] metabric random tabpfn r1f1
[230/330] metabric catboost_rfe tabpfn r1f1 [saved fold reused]
[231/330] metabric no_fs tabpfn r1f1 [saved fold reused]
[232/330] metabric text tabpfn r1f2
[233/330] metabric hybrid tabpfn r1f2
[234/330] metabric adaptive_hybrid tabpfn r1f2
[235/330] metabric data_only tabpfn r1f2
[236/330] metabric mi tabpfn r1f2 [saved fold reused]
[237/330] metabric lasso tabpfn r1f2 [saved fold reused]
[238/330] metabric random tabpfn r1f2
[239/330] metabric random tabpfn r1f2
[240/330] metabric random tabpfn r1f2
[241/330] metabric catboost_rfe tabpfn r1f2 [saved fold reused]
[242/330] metabric no_fs tabpfn r1f2 [saved fold reused]
[243/330] metabric t

status     resume_hit
completed  True          262
           False          68
Name: count, dtype: int64

## 5. OOF 성능·설정 선택·안정성·통계

Random은 세 selector seed의 확률을 표본별 평균합니다. Adaptive Hybrid를 기준으로 모든
비교 방법과 paired bootstrap을 수행합니다.


In [9]:
RUN_METRICS = True
if RUN_METRICS:
    predictions = exp.consolidate_predictions(CFG)
    repeat_metrics = exp.aggregate_repeat_metrics(CFG, predictions)
    fold_metrics = exp.aggregate_fold_metrics(CFG, predictions)
    fold_variability = exp.summarize_fold_variability(CFG, fold_metrics)
    metric_summary = exp.summarize_metrics(CFG, repeat_metrics)
    primary_oof = exp.primary_core_oof_metrics(CFG, predictions)
    tuning_decisions = exp.consolidate_tuning_decisions(CFG)
    stability = exp.compute_selection_stability(CFG)
    selection_frequency = exp.compute_selection_frequency(CFG)
    bootstrap = exp.paired_bootstrap_auroc(CFG, predictions, reference_method='adaptive_hybrid')

    display(primary_oof.sort_values(['dataset', 'sample_regime', 'auroc'], ascending=[True, True, False]))
    display(tuning_decisions.loc[tuning_decisions['stage'] == 'core'].head(30))
    display(bootstrap.sort_values(['dataset', 'sample_regime', 'delta_auroc'], ascending=[True, True, False]))
    assert len(primary_oof) == 54
    assert len(bootstrap) == 48
    complete = stability.loc[(stability['stage'] == 'core') & stability['complete_outer_cv']]
    assert (complete['n_pairs'] == 10).all()


C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3차\EXPERIMENT_CODE\llmfs_pipeline.py:2507: RuntimeWarning: Mean of empty slice
  "mean_kuncheva": float(np.nanmean(kuncheva)) if kuncheva else float("nan"),
C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3차\EXPERIMENT_CODE\llmfs_pipeline.py:2507: RuntimeWarning: Mean of empty slice
  "mean_kuncheva": float(np.nanmean(kuncheva)) if kuncheva else float("nan"),


,dataset,sample_regime,method,auroc,average_precision,ap_lift,log_loss,brier,mcc,balanced_accuracy,sensitivity,specificity,ece,prevalence,n_oof,n_outer_repeats,n_selector_seeds
0,colon,24,adaptive_hybrid,0.852273,0.900071,1.395110,0.490048,0.150743,0.547014,0.776136,0.825000,0.727273,0.091282,0.645161,62,1,1
1,colon,24,catboost_rfe,0.837500,0.909851,1.410269,0.513711,0.163352,0.587147,0.798864,0.825000,0.772727,0.135759,0.645161,62,1,1
6,colon,24,no_fs,0.829545,0.889652,1.378960,0.500370,0.159486,0.608882,0.801136,0.875000,0.727273,0.133719,0.645161,62,1,1
7,colon,24,random,0.829545,0.858154,1.330139,0.490409,0.157210,0.466426,0.730682,0.825000,0.636364,0.111277,0.645161,62,1,3
8,colon,24,text,0.813636,0.890042,1.379565,0.513750,0.168426,0.506818,0.753409,0.825000,0.681818,0.142553,0.645161,62,1,1
3,colon,24,hybrid,0.794318,0.878593,1.361820,0.567498,0.176159,0.517940,0.763636,0.800000,0.727273,0.159015,0.645161,62,1,1
2,colon,24,data_only,0.793182,0.848703,1.315490,0.560020,0.174453,0.506818,0.753409,0.825000,0.681818,0.115255,0.645161,62,1,1
4,colon,24,lasso,0.792045,0.849726,1.317075,0.661035,0.181316,0.466426,0.730682,0.825000,0.636364,0.165033,0.645161,62,1,1
5,colon,24,mi,0.722727,0.796333,1.234316,0.767583,0.221833,0.310317,0.657955,0.725000,0.590909,0.186717,0.645161,62,1,1
9,colon,full,adaptive_hybrid,0.898864,0.943257,1.462049,0.386314,0.119834,0.647727,0.823864,0.875000,0.772727,0.148143,0.645161,62,1,1


,created_at,pipeline_version,stage,dataset,sample_regime,repeat,fold,method,condition,k,...,selected_k,selected_alpha,inner_auroc,inner_log_loss,inner_best_auroc,auroc_tolerance,inner_split_seed,n_train,n_selected,train_sample_hash
0,2026-08-13T15:04:17.296228+00:00,4.1.0,core,colon,24,1,1,adaptive_hybrid,real,128.0,...,64,0.50,0.500,2.415922,0.500,0.005,2567994453,24,64,a29a2d0e027adffbb867ff89df581df741103962451871...
1,2026-08-14T04:28:37.589949+00:00,4.1.0,core,colon,24,1,1,catboost_rfe,name_invariant,128.0,...,128,NaN,0.250,2.111087,0.250,0.005,2567994453,24,128,a29a2d0e027adffbb867ff89df581df741103962451871...
2,2026-08-14T04:28:00.464974+00:00,4.1.0,core,colon,24,1,1,data_only,name_invariant,128.0,...,128,NaN,0.500,2.490693,0.500,0.005,2567994453,24,128,a29a2d0e027adffbb867ff89df581df741103962451871...
3,2026-08-14T04:27:59.170326+00:00,4.1.0,core,colon,24,1,1,hybrid,real,128.0,...,64,0.25,0.250,2.098693,0.250,0.005,2567994453,24,64,a29a2d0e027adffbb867ff89df581df741103962451871...
4,2026-08-14T04:28:23.957389+00:00,4.1.0,core,colon,24,1,1,lasso,name_invariant,128.0,...,32,NaN,0.375,2.114524,0.375,0.005,2567994453,24,32,a29a2d0e027adffbb867ff89df581df741103962451871...
5,2026-08-14T04:28:14.872813+00:00,4.1.0,core,colon,24,1,1,mi,name_invariant,128.0,...,64,NaN,0.375,2.356413,0.375,0.005,2567994453,24,64,a29a2d0e027adffbb867ff89df581df741103962451871...
6,2026-08-14T04:28:25.315387+00:00,4.1.0,core,colon,24,1,1,random,name_invariant,128.0,...,32,NaN,0.750,0.970471,0.750,0.005,2567994453,24,32,a29a2d0e027adffbb867ff89df581df741103962451871...
7,2026-08-14T04:28:25.978881+00:00,4.1.0,core,colon,24,1,1,random,name_invariant,128.0,...,32,NaN,0.500,0.870620,0.500,0.005,2567994453,24,32,a29a2d0e027adffbb867ff89df581df741103962451871...
8,2026-08-14T04:28:24.615387+00:00,4.1.0,core,colon,24,1,1,random,name_invariant,128.0,...,32,NaN,0.500,1.699338,0.500,0.005,2567994453,24,32,a29a2d0e027adffbb867ff89df581df741103962451871...
9,2026-08-14T04:27:57.982327+00:00,4.1.0,core,colon,24,1,1,text,real,128.0,...,128,NaN,0.500,1.689230,0.500,0.005,2567994453,24,128,a29a2d0e027adffbb867ff89df581df741103962451871...


,dataset,sample_regime,n_outer_repeats,metric,reference_method,comparator_method,reference_auroc,comparator_auroc,delta_auroc,ci_level,ci_lower,ci_upper,p_two_sided,bootstrap_iterations,n_samples,p_holm,significant_0_05_holm
4,colon,24,1,auroc,adaptive_hybrid,mi,0.852273,0.722727,1.295455e-01,0.95,0.052349,2.194363e-01,0.001200,5000,62,0.053989,False
3,colon,24,1,auroc,adaptive_hybrid,lasso,0.852273,0.792045,6.022727e-02,0.95,-0.017870,1.456593e-01,0.132773,5000,62,1.000000,False
1,colon,24,1,auroc,adaptive_hybrid,data_only,0.852273,0.793182,5.909091e-02,0.95,-0.009524,1.425618e-01,0.109578,5000,62,1.000000,False
2,colon,24,1,auroc,adaptive_hybrid,hybrid,0.852273,0.794318,5.795455e-02,0.95,-0.016719,1.463436e-01,0.141972,5000,62,1.000000,False
7,colon,24,1,auroc,adaptive_hybrid,text,0.852273,0.813636,3.863636e-02,0.95,-0.036009,1.211760e-01,0.317536,5000,62,1.000000,False
5,colon,24,1,auroc,adaptive_hybrid,no_fs,0.852273,0.829545,2.272727e-02,0.95,-0.030953,7.880792e-02,0.406319,5000,62,1.000000,False
6,colon,24,1,auroc,adaptive_hybrid,random,0.852273,0.829545,2.272727e-02,0.95,-0.034095,8.409938e-02,0.413117,5000,62,1.000000,False
0,colon,24,1,auroc,adaptive_hybrid,catboost_rfe,0.852273,0.837500,1.477273e-02,0.95,-0.025000,5.934633e-02,0.466307,5000,62,1.000000,False
15,colon,full,1,auroc,adaptive_hybrid,text,0.898864,0.844318,5.454545e-02,0.95,0.003174,1.150332e-01,0.036793,5000,62,1.000000,False
10,colon,full,1,auroc,adaptive_hybrid,hybrid,0.898864,0.847727,5.113636e-02,0.95,0.010811,1.025641e-01,0.009198,5000,62,0.386323,False


## 6. 표·시각화 생성 및 완료 검사

공식 성능, calibration, 선택 안정성, inner holdout이 선택한 k·alpha, paired-bootstrap CI와
자원 사용량을 PNG·SVG·PDF·CSV로 저장합니다.


In [10]:
RUN_FIGURES = True
if RUN_FIGURES:
    figure_report = exp.generate_standard_figures(CFG)
    display(figure_report)
    if figure_report['errors']:
        print('Figure warnings:', figure_report['errors'])
    completion = exp.experiment_completion_report(CFG)
    display(completion)
    assert completion.loc[completion['stage'] == 'TOTAL', 'complete'].iloc[0]


C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3차\EXPERIMENT_CODE\llmfs_pipeline.py:2507: RuntimeWarning: Mean of empty slice
  "mean_kuncheva": float(np.nanmean(kuncheva)) if kuncheva else float("nan"),


{'prediction_rows': 44862,
 'repeat_metric_rows': 68,
 'fold_metric_rows': 332,
 'fold_variability_rows': 68,
 'summary_rows': 68,
 'stability_rows': 68,
 'selection_frequency_rows': 22067,
 'primary_oof_metric_rows': 54,
 'paired_bootstrap_rows': 48,
 'compatibility_rows': 0,
 'semantic_rows': 0,
 'nested_tuning_decision_rows': 300,
 'completion': [{'stage': 'smoke',
   'expected': 2,
   'completed': 2,
   'remaining': 0,
   'complete': True},
  {'stage': 'core',
   'expected': 330,
   'completed': 330,
   'remaining': 0,
   'complete': True},
  {'stage': 'TOTAL',
   'expected': 332,
   'completed': 332,
   'remaining': 0,
   'complete': True}],
 'figures': ['C:\\Users\\user\\Desktop\\2026 캡스톤_2차\\2차 실험_KDMS\\3차\\RESULTS_LIGHT_1X5_HOLDOUT\\figures\\core_auroc_vs_sample_size.png',
  'C:\\Users\\user\\Desktop\\2026 캡스톤_2차\\2차 실험_KDMS\\3차\\RESULTS_LIGHT_1X5_HOLDOUT\\figures\\core_auroc_performance_heatmap.png',
  'C:\\Users\\user\\Desktop\\2026 캡스톤_2차\\2차 실험_KDMS\\3차\\RESULTS_LIGHT_1X5_H

,stage,expected,completed,remaining,complete
0,smoke,2,2,0,True
1,core,330,330,0,True
2,TOTAL,332,332,0,True


## 7. 최종 출력 위치

- 공식 성능: `RESULTS_LIGHT_1X5_HOLDOUT/metrics/primary_core_oof_metrics.csv`
- inner 선택 결과: `nested_tuning_decisions.csv`
- paired bootstrap: `paired_bootstrap_auroc.csv`
- 안정성: `selection_stability.csv`, `selection_frequency.csv`
- 그림: `RESULTS_LIGHT_1X5_HOLDOUT/figures`
- 오류와 실행 로그: `RESULTS_LIGHT_1X5_HOLDOUT/logs`


In [11]:
print('Experiment output root:', CFG['results_root'])
print('Primary metrics:', PATHS['metrics'] / 'primary_core_oof_metrics.csv')
print('Nested decisions:', PATHS['metrics'] / 'nested_tuning_decisions.csv')
print('Paired bootstrap:', PATHS['metrics'] / 'paired_bootstrap_auroc.csv')
print('Figures:', PATHS['figures'])


Experiment output root: C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3차\RESULTS_LIGHT_1X5_HOLDOUT
Primary metrics: C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3차\RESULTS_LIGHT_1X5_HOLDOUT\metrics\primary_core_oof_metrics.csv
Nested decisions: C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3차\RESULTS_LIGHT_1X5_HOLDOUT\metrics\nested_tuning_decisions.csv
Paired bootstrap: C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3차\RESULTS_LIGHT_1X5_HOLDOUT\metrics\paired_bootstrap_auroc.csv
Figures: C:\Users\user\Desktop\2026 캡스톤_2차\2차 실험_KDMS\3차\RESULTS_LIGHT_1X5_HOLDOUT\figures
